# HW2 Feature Implementation

## Required Features (Section 5)

### 1. System Prompt — All Six Layers
The system prompt is split into a shared base ([`_SHARED_PROMPT`](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/app.py#L64)) and two persona-specific prompts ([`PROMPT_A`](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/app.py#L117), [`PROMPT_B`](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/app.py#L145)). Together they cover all six required layers: identity and role (Dr. Elena/Edward, 15 years clinical practice, Beck Institute); theoretical framework (cognitive model, [10 distortions](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/app.py#L67), Socratic questioning, thought records, behavioural activation, homework setting); session structure (six-step arc, strict for Dr. Elena, loose for Dr. Edward); communication style (precise and directive vs. warm and unhurried); safety guardrails ([crisis protocol](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/app.py#L79) with language-specific helplines, explicit carve-out for dreams and intrusive thoughts); and scope limitations (no diagnosis, no direct advice, redirect off-topic). Two few-shot example exchanges and explicit `NEVER` prohibitions are included per persona.

### 2. Multi-Turn Conversation with Persistent History
Full session history is stored in [SQLite](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/database.py) and passed to the LLM on every turn. Sessions are reloadable from the sidebar.

### 3. Automatic Opening Message
Each persona has a [dedicated opening message](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/app.py#L180) injected as the first assistant turn. It introduces the doctor by name, states the AI disclaimer, and opens with the mood check-in question.

### 4. Crisis Response
The [safety protocol](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/app.py#L79) triggers on active suicidal ideation or self-harm. It acknowledges the patient's pain, states the tool's limitations, and provides crisis resources in the patient's language. Dreams and intrusive thoughts are [explicitly excluded](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/app.py#L90) from triggering it — dark content in the context of dreams or nightmares is instead treated as legitimate therapeutic material and explored with Socratic questioning.

### 5. Streaming Output
Responses stream token-by-token via `ollama.chat(..., stream=True)`. A [`@st.fragment(run_every=0.1)`](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/app.py#L750) fragment polls a shared buffer and updates the display in real time.

### 6. Session Reset
["End session and start over"](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/app.py#L307) clears all state, creates a new conversation with a history-enriched system prompt, and re-injects the opening message.

### 7. UI Disclaimer
A static disclaimer appears in the [sidebar footer](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/app.py#L610). The opening message also states the tool is not a licensed therapist.

---

## Extensions (Section 6)

### Session Summary Export ✅
["Generate session summary"](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/app.py#L454) produces a structured LLM-generated summary (Main Themes, Cognitive Distortions Identified, Homework Assigned, Key Insights) at temperature 0.3, saves it to the database, and offers a `.txt` download.

### Mood Tracking ✅
The check-in step asks for a 1–10 rating. The app [extracts the first such number](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/app.py#L899) from the patient's messages via regex, stores it in a [`mood_ratings` table](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/database.py#L33), and displays a line chart across sessions in the sidebar.

### Prompt A/B Testing ✅ *(adapted)*
Implemented as two fully developed selectable personas — Dr. Elena (structured/directive) and Dr. Edward (Socratic/collaborative) — rather than running identical inputs through two prompts and scoring them. A [pre-session screen](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/app.py#L616) lets the user choose before each conversation begins.

### Multi-Language Support ✅
The [system prompt](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/app.py#L115) instructs the therapist to mirror the patient's language across every part of every response, including crisis messages and helpline numbers. The safety protocol branches explicitly by language: English patients receive Crisis Text Line and Samaritans; Spanish patients receive Teléfono de la Esperanza and SAPTEL. The STT language selector passes a language hint to Whisper; the TTS uses a [dedicated Spanish VITS model](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/tts.py#L8) when Coqui is selected.

### Thought Record Form ✗
Not implemented.

---

## Optional Features (Section 8)

### 8.1 Voice Interface ✅
Both directions are fully implemented using local, open-source tools — no audio leaves the machine.

- **STT ([`stt.py`](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/stt.py)):** Whisper [`large-v3`](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/stt.py#L12) on CPU, non-blocking `sounddevice` stream with start/stop buttons, [silence gate](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/stt.py#L51) (mean amplitude < 0.001) to prevent hallucination on empty audio, `fp16=False` for stability, deferred transcription to avoid blocking the UI, and a language selector (English/Spanish).
- **TTS ([`tts.py`](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/tts.py)):** Three selectable backends — macOS `say` (zero setup, default voices Samantha/Monica), [Kokoro](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/tts.py#L42) (lightweight ~80 MB, default), and Coqui XTTS v2 (voice cloning with optional reference WAV upload, plus a dedicated `tts_models/es/css10/vits` model for native Spanish without a reference). Audio runs in a background thread after each response and is discarded immediately after playback, consistent with the PDF's transcript-only storage principle.

### 8.3 Bot-to-Bot Patient Testing ✅ *(partially)*
Not yet in this commit — implemented in the subsequent commit. The architecture is in place: mode selector in sidebar, 12-patient grid, turn controls, diagnostic overlay on the system prompt, and result comparison against `/session/reveal`. Currently blocked by a connectivity issue with the remote API.

---

## Additional (Beyond PDF Scope)

**Cross-Session History Injection:** [`build_system_prompt_with_history()`](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/app.py#L205) retrieves the 10 most recent sessions with summaries and mood ratings and injects them as a structured `PATIENT HISTORY` block in the system prompt at session start, giving the therapist continuity across conversations.

**Input Filter:** A secondary LLM ([`checker.py`](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/checker.py#L26)) screens each patient message [before the main model is called](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/app.py#L683). Non-therapeutic inputs are classified as `REDIRECT` and receive a warm one-sentence reply in the patient's language — without ever reaching the therapist LLM.

**File Attachment:** Patients can attach documents (PDF, DOCX, TXT, MD, CSV) to any message. Text is [extracted](https://github.com/asugar13/local-llm/blob/7eaac6dbfa3f65dff45f1c688c2271463d3da145/app.py#L19) and prepended to the LLM context. The attachment persists across sidebar-triggered reruns via session state, shown as a `📎 filename [✕]` indicator.